In [9]:
import pandas as pd
from pathlib import Path
import re


BASE_DIR = '03_articles'
SOURCE_CSV = 'coffee_price_source_websites.csv'
OUTPUT_CSV = 'coffee_articles.csv'

In [10]:
def load_articles_from_folder(base_dir):
    """Quét thư mục và trả về DataFrame chứa nội dung bài viết."""
    print(f"Đang quét các file trong thư mục {base_dir}...")
    data_list = []
    
    for file_path in Path(base_dir).rglob('*'):
        if file_path.is_file() and not file_path.name.startswith('.'):
            try:
                with open(file_path, 'r', encoding='utf-8') as f:
                    data_list.append({
                        'PUBLISHED_DATE': file_path.parent.name,
                        'FILENAME': file_path.name.replace('.html', ''), 
                        'CONTENT': f.read(),
                        'FILE_PATH': str(file_path)
                    })
            except Exception as e:
                print(f"Lỗi khi đọc file {file_path}: {e}")

    df_articles = pd.DataFrame(data_list)
    print(f"Hoàn tất! Đã load thành công {len(df_articles)} file.\n" + "-" * 30)
    
    return df_articles

In [11]:
def merge_with_source_data(df_articles, source_csv):
    
    df = pd.read_csv(source_csv)
    
    
    df_result = df[df['URL_HASH'].isin(df_articles['FILENAME'])].drop_duplicates(subset=['URL_HASH'])
    
    
    df_combined = pd.merge(df_result, df_articles[['FILENAME', 'CONTENT']], 
                           left_on='URL_HASH', right_on='FILENAME', how='inner')
    
    cols_to_drop = ['FILENAME', 'SEARCH_QUERY', 'TITLE', 'SCRAPE_STATUS', 'DATE_REF', 'SNIPPET']
    df_combined = df_combined.drop(columns=cols_to_drop)
    
    return df_combined

In [ ]:
def filter_coffee_and_export(df_combined, output_csv):
    
    key_words = ['robusta', 'arabica', 'giá cà phê']
    pattern = '|'.join(key_words)
  
    df_combined['IS_COFFEE'] = df_combined['CONTENT'].apply(
        lambda x: 1 if pd.notnull(x) and re.search(pattern, x, re.IGNORECASE) else 0
    )
    
   
    df_final = df_combined[df_combined['IS_COFFEE'] == 1].drop(columns=['IS_COFFEE'])
    
    
    df_final.to_csv(output_csv, index=False)
    print(f"Đã xuất dữ liệu thành công ra file: {output_csv}")
    
    return df_final 

In [13]:

df_articles = load_articles_from_folder(BASE_DIR)


if not df_articles.empty:
    df_combined = merge_with_source_data(df_articles, SOURCE_CSV)
    df_final = filter_coffee_and_export(df_combined, OUTPUT_CSV)
    
    
    
else:
    print("Không tìm thấy bài viết nào để xử lý.")

Đang quét các file trong thư mục 03_articles...
Hoàn tất! Đã load thành công 11769 file.
------------------------------
Đã xuất dữ liệu thành công ra file: coffee_articles.csv
